# 1. Exploração e preparação dos dados

**Equipe:** Diogo Galrão Carvalho; Felipe Artur Macedo Lima; Luan Cavalcante Dias Rodrigues.

**Disciplina:** Banco de Dados e Big Data para Data Science, Pós-Graduação em Data Science e Analytics.

O catálogo foi importado em CSV. Partidas, compras e demografia são sintéticas, com seed 42. Os nicks são ilustrativos e não representam observações sobre jogadores reais. Seis itens gratuitos recebem preço simbólico de 150 ouro; esse ajuste é preservado para manter comparabilidade com a base original.

In [1]:
from pathlib import Path
import sys
import os
ROOT = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import pandas as pd
raw = pd.read_csv(ROOT / "data/raw/itens_raw.csv")
print(f"Catálogo bruto: {len(raw)} itens")
raw[["item_id", "nome", "preco_total_ouro", "categorias_json"]].head()

Catálogo bruto: 254 itens


,item_id,nome,preco_total_ouro,categorias_json
0,1001,Botas,300,"[""Boots""]"
1,1004,Amuleto da Fada,200,"[""ManaRegen""]"
2,1006,Pérola do Rejuvenescimento,300,"[""HealthRegen""]"
3,1011,Cinto do Gigante,900,"[""Health""]"
4,1018,Capa da Agilidade,600,"[""CriticalStrike""]"


## Limpeza e geração reproduzível

A geração está em `scripts/generate_harmonized_data.py`: categorias JSON são expandidas para uma tabela N:N; itens são deduplicados por ID; compras são simuladas dentro da duração de cada partida. Executar esta célula reconstitui os CSVs da amostra.

In [2]:
from scripts.generate_harmonized_data import main
main()

==> 1. Processando itens reais e categorias...
    - Categorias extraídas: 32
    - Itens extraídos: 254
    - Relacionamentos Item-Categoria (N:N): 834
==> 2. Gerando Elos...
==> 3. Gerando Jogadores e Demografia Complementar (para DBFS / Spark)...
    - Jogadores: 25
    - Registros demográficos complementares (DBFS): 25
==> 4. Gerando Partidas...
    - Partidas geradas: 30
==> 5. Gerando Compras...
    - Compras geradas: 1043

==> 6. Exportando arquivos para data/processed/ ...
    [OK] Salvo: data/processed/elo.csv (10 linhas)
    [OK] Salvo: data/processed/categoria.csv (32 linhas)
    [OK] Salvo: data/processed/item.csv (254 linhas)
    [OK] Salvo: data/processed/item_categoria.csv (834 linhas)
    [OK] Salvo: data/processed/jogador.csv (25 linhas)
    [OK] Salvo: data/processed/jogadores_demografia.csv (25 linhas)
    [OK] Salvo: data/processed/partida.csv (30 linhas)
    [OK] Salvo: data/processed/compra.csv (1043 linhas)

[OK] Concluído! Todos os dados relacionais e complement

## Verificação de chaves e valores

O mesmo validador da migração rejeita chaves duplicadas, referências órfãs, totais incoerentes e compras fora da partida. A coluna `total_compra` armazena um valor derivado. O validador confere essa redundância, que impede classificar todos os atributos como estritamente normalizados.

In [3]:
from scripts.migrate_to_mongodb import load_tables, build_documents
tables = load_tables("csv")
documents = build_documents(tables)
display(pd.DataFrame([{"tabela": k, "linhas": len(v)} for k, v in tables.items()]))
print("Compras:", sum(j["total_transacoes"] for j in documents["jogadores"]))
print("Ouro:", sum(j["total_gasto_ouro"] for j in documents["jogadores"]))

,tabela,linhas
0,elo,10
1,categoria,32
2,item,254
3,item_categoria,834
4,jogador,25
5,partida,30
6,compra,1043


Compras: 1043
Ouro: 2228821


## Limites

A amostra demonstra modelagem e transformações. Não mede o comportamento real do jogo nem desempenho em grande escala. DDL e carga PostgreSQL estão em `docker/init-db`; o notebook seguinte lê e verifica os dados do PostgreSQL.